# Consistency: Countdown SFT + discrimination test (3B) (first results)

The notebook finetunes Llama-3.2-3B-Instruct with LoRA on algorithmically produced search traces for puzzles sampled from TinyZero 4-number puzzles. 
Dependencies: `countdown.py` (solver, checker) and `traces.py` (prompt, program-written traces) from the repo's `countdown/`.

In [ ]:
# [config]
import os
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
# Policy (samples the completions):
#   "countdown_sft"  LoRA from [SFT train] on program-written Countdown traces (traces.py), trained from the base model
#   "countdown_sft_dpo"  the same after [judge DPO]
#   "base"           no adapter
POLICY = "countdown_sft"

N_SFT = 3000                    # N of 4-number traces to use for SFT
N_SFT_3 = 3000                  # N of 3-number traces to use for SFT
N_SFT_VAL = 100                 # n to estimate validation loss, greedy accuracy
SFT_EPOCHS = 1
SFT_LR = 1e-4
LORA_R, LORA_ALPHA = 16, 32
SFT_BATCH, SFT_GRAD_ACC = 4, 4
SFT_EVAL_STEPS = 25
RETRAIN = False
TRACE_VERSION = 2               # traces.py: 1 = the first run (checks always pass, no "Left:" on the winning route);
                                # 2 = "Left:" on every step, false finishes caught by the check, "Tried:" after back-ups


# Judge trains on the same SFT adapter. Judge format = neutral system prompt + bare puzzle -> solution + "Answer: ..."

#   N_JUDGE_SFT  gold judge-format examples mixed into [SFT data] (one per SFT puzzle, the first N); 0 = traces only.
#                Changes the SFT run name (a new adapter)
#   JUDGE_DPO    [judge DPO]: continue the SFT adapter with DPO on judge-format pairs, chosen = the gold solution,
#                rejected = a wrong route of its own, wrong in one step anywhere: swap (a random unavailable
#                number, arithmetic right) or slip (one false result, the rest done right from it), and with
#                "miss" in DPO_KINDS an honest route that ends off the target (every step true; the pass arm's
#                exploit on the unfiltered sample, 2026-10-17);
#                reference = the SFT adapter (TRL copies it). The result is the "dpo" judge (JUDGES) and the
#                "countdown_sft_dpo" policy
N_JUDGE_SFT = 3000              # half from each size with N_SFT_3
JUDGE_DPO = True
N_DPO = 2250                    # pairs, from the SFT training puzzles (validation pairs from the SFT validation puzzles)
N_DPO_3 = 2250                  # + pairs from 3-number puzzles of the SFT draw
DPO_KINDS = ("swap", "slip", "miss")   # rejected kinds, in turn per puzzle; ("swap", "slip") = the 2026-10-16 judge
DPO_SHARES = {"swap": 0.3, "extra": 0.3, "slip": 0.2, "miss": 0.2}   # quotas of N_DPO and N_DPO_3 per kind instead
                                # (DPO_KINDS unused); None = in turn. swap and extra (a number too many: the policy's
                                # main 3-number error) weigh more: invented / reused numbers are the exploits
# Real pairs: chosen = a right, rejected = a wrong completion of the SFT policy on the same puzzle (judge format, as the
# GRPO reward scores them), from fresh puzzles ([puzzles]) sampled in [real samples]; at most REAL_PER_PUZZLE per
# puzzle; the last 10% of the puzzles give the validation pairs. 0 = none
N_REAL_4, N_REAL_3 = 0, 0          # (1000, 500): the judge with real pairs (2026-10-19); 0, 0 = solver-made pairs only
REAL_PUZZLES_4, REAL_PUZZLES_3 = 1100, 500
REAL_PER_PUZZLE = 2
# REAL_USE_4 / _3: (puzzles, samples per puzzle) of the sampled training puzzles that the real pairs may come from (the
# first ones; no pair cap then); None = all (990 x 16 / 450 x 16).
REAL_USE_4, REAL_USE_3 = (500, 16), (250, 8)
DPO_BETA, DPO_LR, DPO_EPOCHS = 0.1, 2e-5, 1
RETRAIN_DPO = False
# Judges score every completion: "base" = the base model (adapter off; the judge of the 3B GSM8K GRPO runs),
# "policy" = the policy itself (the adapter on; skipped when POLICY = "base"), "dpo" = the [judge DPO] adapter
# (skipped until it exists). [score] rescores saved samples when a judge is added.
JUDGES = ("base", "policy", "dpo")
N_PUZZLES = 200                 # TinyZero 4-number puzzles (Jiayi-Pan/Countdown-Tasks-3to4)
SAMPLES = 16                    # completions per puzzle (a GRPO group)
EVAL_PUZZLES = 100              # [sample]: the first EVAL_PUZZLES of each size (3, 4, 5) are sampled (200 up to 2026-10-21)
TEMPERATURE = 0.9               # as in GRPO
MAX_TOKENS = 1024
PROMPT_STYLE = "user"           # "user": puzzle + format block (the SFT prompt); "one_shot": plus one program-written
                                # trace (other puzzle) as an example turn, for policies that never saw Countdown
SEED = 0
VLLM_GPU_MEMORY_UTILIZATION = 0.5
HF_CHECK = True                 # [score]: compare a few vLLM scores with a plain transformers forward pass

DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
SFT_RUN_NAME = (f"{MODEL.split('/')[-1]}-countdown{N_SFT}-ep{SFT_EPOCHS}-lr{SFT_LR:g}-seed{SEED}"
                + (f"-tr{TRACE_VERSION}" if TRACE_VERSION != 1 else "")    # version 1 keeps the first run's folder
                + (f"-judge{N_JUDGE_SFT}" if N_JUDGE_SFT else "") + (f"-mix3x{N_SFT_3}" if N_SFT_3 else ""))
SFT_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN_NAME}"
_real_cut = bool(N_REAL_4 or N_REAL_3) and (REAL_USE_4 is not None or REAL_USE_3 is not None)      # in the judge's name ("_use..."); all = the old name
_real_use = lambda use, n: (n - n // 10, SAMPLES) if use is None else use    # the training part: all but the last 10%
REAL_USE_4, REAL_USE_3 = _real_use(REAL_USE_4, REAL_PUZZLES_4), _real_use(REAL_USE_3, REAL_PUZZLES_3)
REAL_LABELS = (N_REAL_4 > 0) * REAL_USE_4[0] * REAL_USE_4[1] + (N_REAL_3 > 0) * REAL_USE_3[0] * REAL_USE_3[1]
DPO_DIR = (f"{SFT_DIR}/judge_dpo_" + ("".join(f"{k}{round(100 * v)}" for k, v in DPO_SHARES.items()) if DPO_SHARES
                                       else "".join(DPO_KINDS))    # swapslip n3000: the 2026-10-16 judge
           + f"_n{N_DPO}" + (f"_n3x{N_DPO_3}" if N_DPO_3 else "")
           + (f"_real{N_REAL_4}x{N_REAL_3}" if N_REAL_4 or N_REAL_3 else "")
           + (f"_use{REAL_USE_4[0]}x{REAL_USE_4[1]}_{REAL_USE_3[0]}x{REAL_USE_3[1]}" if _real_cut else "")
           + f"_b{DPO_BETA:g}_lr{DPO_LR:g}_ep{DPO_EPOCHS}")
REAL_SAMPLES_PATH = f"{SFT_DIR}/real_samples_n4x{REAL_PUZZLES_4}_n3x{REAL_PUZZLES_3}_g{SAMPLES}_t{TEMPERATURE:g}_seed{SEED}.csv"
DPO_ADAPTER_DIR = f"{DPO_DIR}/adapter"
_DPO_OLD = "judge_dpo_swapslip_n3000_b0.1_lr2e-05_ep1"
DPO_TAG = "" if os.path.basename(DPO_DIR) == _DPO_OLD else "_" + os.path.basename(DPO_DIR)   # in the file names of saved "dpo" scores
assert POLICY in ("countdown_sft", "countdown_sft_dpo", "gsm8k_sft", "base"), POLICY
ADAPTER_PATH = {"countdown_sft": f"{SFT_DIR}/adapter", "countdown_sft_dpo": DPO_ADAPTER_DIR,
                "gsm8k_sft": f"{DRIVE}/sft/{GSM8K_SFT_RUN}/adapter", "base": None}[POLICY]
_policy = {"countdown_sft": SFT_RUN_NAME, "countdown_sft_dpo": f"{SFT_RUN_NAME}-{os.path.basename(DPO_DIR)}",
           "gsm8k_sft": f"{GSM8K_SFT_RUN}-adapter", "base": MODEL.split("/")[-1]}[POLICY]
OUT_DIR = f"{DRIVE}/countdown/disc/{_policy}-{PROMPT_STYLE}-n{EVAL_PUZZLES}x{SAMPLES}-t{TEMPERATURE:g}-seed{SEED}"
assert set(JUDGES) <= {"base", "policy", "dpo"}, JUDGES
print(f"MODEL={MODEL}  POLICY={POLICY}  adapter={ADAPTER_PATH}  judges={JUDGES}\n{N_PUZZLES} puzzles x {SAMPLES}  "
      f"prompt={PROMPT_STYLE}\nOUT_DIR={OUT_DIR}")

MODEL=meta-llama/Llama-3.2-3B-Instruct  POLICY=countdown_sft  adapter=/content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000/adapter  judges=('base', 'policy', 'dpo')
200 puzzles x 16  prompt=user
OUT_DIR=/content/drive/MyDrive/Consistency/countdown/disc/Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000-user-n100x16-t0.9-seed0


In [ ]:
# [drive + HF] Mount Drive and log in to HF (Colab Secret HF_TOKEN).
import os
from google.colab import drive, userdata
from huggingface_hub import login
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
login(userdata.get("HF_TOKEN"))
assert os.path.exists(f"{CODE_DIR}/countdown.py"), f"{CODE_DIR}/countdown.py missing (copy countdown/ from the repo)"
assert POLICY != "gsm8k_sft" or os.path.exists(f"{ADAPTER_PATH}/adapter_model.safetensors"), f"no adapter at {ADAPTER_PATH}"

In [ ]:
# [install] 
import importlib.util, os, subprocess, sys
if importlib.util.find_spec("vllm") and importlib.util.find_spec("trl"):
    print("[install] skipped: vllm and trl are installed")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl[vllm]", "peft"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] done: restarting the session; Run all again")
    os.kill(os.getpid(), 9)

[install] skipped: vllm and trl are installed


In [ ]:
# [vLLM setup]
import os, sys
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the vLLM engine in this process
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):   # Colab's streams have no file descriptor; vLLM asks for one
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import importlib.metadata as md
for p in ["torch", "transformers", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")

name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB
torch          2.11.0+cu130
transformers   5.20.1
datasets       4.8.5
vllm           0.26.0


In [ ]:
# [puzzles] Tokenizer, TinyZero 4-number puzzles (seeded draw: test puzzles, the one-shot example, and apart from those
# the SFT puzzles), the number of distinct solution expressions per test puzzle (solver; fewer = harder), the prompt,
# and the answer/step checks.
import os, sys, re, json, random, importlib
from fractions import Fraction
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = "<|finetune_right_pad_id|>"   # not <|eot_id|> (the EOS); as in the SFT notebook
tok.chat_template = re.sub(r'strftime_now\([^)]*\)', '"26 Jul 2024"', tok.chat_template)   # pinned date

def chat_ids(messages):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tok(text, add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)
import countdown, traces, judge_data
importlib.reload(countdown); importlib.reload(traces); importlib.reload(judge_data)
from countdown import check_answer, solve
from traces import countdown_prompt, countdown_question, make_trace

_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_rng = np.random.default_rng(SEED)
_draw = _tz.select(_rng.choice(len(_tz), size=min(len(_tz), 20 * N_PUZZLES), replace=False))
_four = [{"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])} for r in _draw if len(r["nums"]) == 4]
assert len(_four) > N_PUZZLES, f"only {len(_four)} 4-number puzzles in the draw"
EXAMPLE, puzzles = _four[0], _four[1:N_PUZZLES + 1]      # the one-shot example is a different puzzle
for p in puzzles:
    p["n_solutions"] = len(solve(p["numbers"], p["target"]))
assert all(p["n_solutions"] > 0 for p in puzzles), "unsolvable puzzle (solver or data problem)"
# The judge evals also cover other puzzle sizes: the test draw's first N_PUZZLES
# 3-number puzzles (Consistency_GRPO_Countdown's 3-number test set; in no SFT, DPO or GRPO training data) and N_PUZZLES
# generated 5-number puzzles (TinyZero has 3 and 4 only, all solvable by construction)
_p3 = [{"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])} for r in _draw if len(r["nums"]) == 3][:N_PUZZLES]
for p in _p3:
    p["n_solutions"] = len(solve(p["numbers"], p["target"]))
_r5 = random.Random(SEED + 5)
_p5 = [{k: g[k] for k in ("numbers", "target", "n_solutions")}
       for g in (countdown.generate(_r5, n_numbers=5, count_solutions=False) for _ in range(N_PUZZLES))]
puzzles_by_size = {4: puzzles, 3: _p3, 5: _p5}
# SFT puzzles: a separate seeded draw (the test draw above is unchanged), without the test puzzles and the example
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])
_seen = {_key(p) for p in puzzles} | {_key(EXAMPLE)}
_draw2 = _tz.select(np.random.default_rng(SEED + 1).choice(len(_tz), size=min(len(_tz), 6 * (4000 + 100)),
                                                           replace=False))   # fixed size: GRPO excludes this whole draw
sft_puzzles = []
for r in _draw2:
    p = {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
    if len(p["numbers"]) == 4 and _key(p) not in _seen:
        _seen.add(_key(p))
        sft_puzzles.append(p)
assert len(sft_puzzles) >= N_SFT + N_SFT_VAL, f"only {len(sft_puzzles)} SFT puzzles in the draw"
sft_puzzles = sft_puzzles[:N_SFT + N_SFT_VAL]
# 3-number judge puzzles ([judge DPO], N_DPO_3): the 3-number puzzles of the same SFT draw, none from the test draw
# (Consistency_GRPO_Countdown's 3-number test set comes from it; its 3-number training puzzles exclude this whole draw)
_test_draw = {_key({"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}) for r in _draw}
dpo3_puzzles = []
for r in _draw2:
    p = {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
    if len(p["numbers"]) == 3 and _key(p) not in _seen and _key(p) not in _test_draw:
        _seen.add(_key(p))
        dpo3_puzzles.append(p)
assert len(dpo3_puzzles) >= max(N_DPO_3, N_SFT_3) + 2 * N_SFT_VAL, f"only {len(dpo3_puzzles)} 3-number puzzles in the SFT draw"
dpo3_puzzles = dpo3_puzzles if N_DPO_3 or N_SFT_3 else []
# with N_SFT_3: SFT traces [:N_SFT_3], validation the next N_SFT_VAL (SFT and DPO), DPO pairs from the start (quotas
# go on after the validation puzzles); without: DPO train [:N_DPO_3], validation the next 100, quotas go on after
sft3_puzzles, sft3_val = dpo3_puzzles[:N_SFT_3], dpo3_puzzles[N_SFT_3:N_SFT_3 + N_SFT_VAL] if N_SFT_3 else []
# Real-pair puzzles ([real samples]): a fifth seeded draw (SEED + 4) without every puzzle of the test and SFT draws and of
# Consistency_GRPO_Countdown's RL draw (SEED + 2) and 3-number training draw (SEED + 3) at its defaults (POOL_CANDIDATES
# 4000, UNSCORED_N 100, TRAIN_MIX_3 0.5: 3900 3-number puzzles): REAL_PUZZLES_4 4-number, then REAL_PUZZLES_3 3-number
_as = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_seeded = lambda seed, size: _tz.select(np.random.default_rng(seed).choice(len(_tz), size=min(len(_tz), size), replace=False))
real_puzzles = []
if N_REAL_4 or N_REAL_3:
    _no_real = (_seen | _test_draw | {_key(_as(r)) for r in _draw2}
                | {_key(_as(r)) for seed, size in ((SEED + 2, 4 * (4000 + 100)), (SEED + 3, 4 * 3900)) for r in _seeded(seed, size)})
    _real = {4: [], 3: []}
    for r in _seeded(SEED + 4, 10 * (REAL_PUZZLES_4 + REAL_PUZZLES_3)):
        p = _as(r)
        if _key(p) not in _no_real and len(_real[len(p["numbers"])]) < {4: REAL_PUZZLES_4, 3: REAL_PUZZLES_3}[len(p["numbers"])]:
            _no_real.add(_key(p))
            _real[len(p["numbers"])].append(p)
    assert len(_real[4]) == REAL_PUZZLES_4 and len(_real[3]) == REAL_PUZZLES_3, {k: len(v) for k, v in _real.items()}
    real_puzzles = _real[4] + _real[3]
EXAMPLE_TRACE = make_trace(EXAMPLE["numbers"], EXAMPLE["target"], random.Random(SEED), n_dead_ends=1)["completion_text"]

def build_prompt(numbers, target):
    if PROMPT_STYLE == "one_shot":
        return (countdown_prompt(EXAMPLE["numbers"], EXAMPLE["target"])
                + [{"role": "assistant", "content": EXAMPLE_TRACE}] + countdown_prompt(numbers, target))
    return countdown_prompt(numbers, target)

def tag(text, name):
    """Content of the last <name>...</name> block, or None."""
    m = re.findall(rf"<{name}>(.*?)</{name}>", text, re.DOTALL)
    return m[-1].strip() if m else None

_NUM = r"\(?\s*(-?\d+(?:\s*/\s*\d+)?)\s*\)?"
_STEP = re.compile(rf"^\s*{_NUM}\s*([+\-*/×÷x])\s*{_NUM}\s*=\s*{_NUM}\s*\.?\s*$")

def arithmetic_ok(solution):
    """True/False if every non-empty line of the solution is 'a op b = c' and each is (in)correct; None if any line
    isn't in that form (unknown)."""
    lines = [l for l in (solution or "").splitlines() if l.strip()]
    if not lines:
        return None
    for line in lines:
        m = _STEP.match(line)
        if not m:
            return None
        a, op, b, c = m.groups()
        a, b, c = (Fraction(s.replace(" ", "")) for s in (a, b, c))
        if op in "/÷" and b == 0:
            return False
        r = {"+": a + b, "-": a - b, "*": a * b, "×": a * b, "x": a * b}.get(op, a / b if b else None)
        if r != c:
            return False
    return True

import ast
from collections import Counter

def _numbers_in(expr):
    """Integer literals of an answer that parses as arithmetic (the checker's normalization), "decimals", or None."""
    e = expr.strip().replace("×", "*").replace("÷", "/").replace("−", "-").split("=")[0]
    try:
        tree = ast.parse(e, mode="eval")
    except SyntaxError:
        return None
    if not all(isinstance(n, (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.operator, ast.unaryop))
               for n in ast.walk(tree)):
        return None
    vals = [n.value for n in ast.walk(tree) if isinstance(n, ast.Constant)]
    return vals if all(isinstance(v, int) and not isinstance(v, bool) for v in vals) else "decimals"

def diagnose(answer, numbers, target, truncated):
    """What is wrong with an answer (first matching): no answer (truncated) / no answer / correct / not an expression /
    uses decimals / wrong numbers (made up or swapped) / a number reused or added / a number missing / right numbers,
    wrong value."""
    if not isinstance(answer, str) or not answer.strip():
        return "no answer (truncated)" if truncated else "no answer"
    if check_answer(answer, numbers, target):
        return "correct"
    nums = _numbers_in(answer)
    if nums is None:
        return "not an expression"
    if nums == "decimals":
        return "uses decimals"
    extra, missing = Counter(nums) - Counter(numbers), Counter(numbers) - Counter(nums)
    if extra and missing:
        return "wrong numbers (made up or swapped)"
    if extra:
        return "a number reused or added"
    if missing:
        return "a number missing"
    return "right numbers, wrong value"

_ns = pd.Series([p["n_solutions"] for p in puzzles])
print(f"{len(puzzles)} test puzzles, {len(sft_puzzles)} SFT puzzles, {len(dpo3_puzzles)} 3-number DPO puzzles, "
      f"{len(real_puzzles)} real-pair puzzles (disjoint); test distinct solutions per puzzle: median {_ns.median():.0f}, "
      f"1: {(_ns == 1).mean():.2f}, <=4: {(_ns <= 4).mean():.2f}")
print("example puzzle:", countdown_question(puzzles[0]["numbers"], puzzles[0]["target"]))
print("one-shot trace:\n" + EXAMPLE_TRACE if PROMPT_STYLE == "one_shot" else "")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.85MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

200 test puzzles, 3100 SFT puzzles, 11797 3-number DPO puzzles, 0 real-pair puzzles (disjoint); test distinct solutions per puzzle: median 5, 1: 0.15, <=4: 0.43
example puzzle: Using the numbers 2, 14, 70, 12, create an expression that equals 57. You can use +, -, * and /, and each number must be used exactly once.



In [ ]:
# [SFT data] One program-written trace per SFT puzzle (traces.py, TRACE_VERSION: 0-3 dead ends, "Back up.", the
# working route, a check; version 2 adds "Left:" on every step, false finishes the check rejects, "Tried:"; solution =
# one step per line; answer = the expression), prompt = the puzzle + format block (PROMPT_STYLE "user"). The last
# N_SFT_VAL traces (per size) give the validation loss. With N_JUDGE_SFT, gold judge-format
# examples (judge_data.py) are mixed into the training set (SFT_DIR/judge_sft_data.csv).
from datasets import Dataset

_trng = random.Random(SEED)
sft_traces = [make_trace(p["numbers"], p["target"], _trng, version=TRACE_VERSION) for p in sft_puzzles]
sft3_traces = [make_trace(p["numbers"], p["target"], _trng, version=TRACE_VERSION) for p in sft3_puzzles + sft3_val]
assert all(check_answer(t["answer"], t["numbers"], t["target"]) for t in sft_traces + sft3_traces), "a trace answer fails the checker"
_as_row = lambda t: {"prompt": countdown_prompt(t["numbers"], t["target"]),
                     "completion": [{"role": "assistant", "content": t["completion_text"]}]}
_rows, _rows3 = [_as_row(t) for t in sft_traces], [_as_row(t) for t in sft3_traces]
if not N_JUDGE_SFT:
    _judge_rows = []
elif N_SFT_3:   # half from each size
    _judge_rows = judge_data.judge_examples(sft_puzzles[:N_JUDGE_SFT // 2] + sft3_puzzles[:N_JUDGE_SFT - N_JUDGE_SFT // 2],
                                            random.Random(SEED + 3))
else:
    _judge_rows = judge_data.judge_examples(sft_puzzles[:min(N_JUDGE_SFT, N_SFT)], random.Random(SEED + 3))
_train_rows = _rows[:N_SFT] + _rows3[:N_SFT_3] + _judge_rows
random.Random(SEED + 4).shuffle(_train_rows)    # traces (both sizes) and judge examples mixed
sft_dataset, sft_val_dataset = Dataset.from_list(_train_rows), Dataset.from_list(_rows[N_SFT:] + _rows3[N_SFT_3:])
os.makedirs(SFT_DIR, exist_ok=True)
pd.DataFrame(sft_traces + sft3_traces).assign(
    split=["train"] * N_SFT + ["val"] * N_SFT_VAL + ["train"] * N_SFT_3 + ["val"] * len(sft3_val)).to_csv(f"{SFT_DIR}/sft_data.csv", index=False)
_ntok = pd.Series([len(tok(t["completion_text"], add_special_tokens=False)["input_ids"]) for t in sft_traces + sft3_traces])
if _judge_rows:
    pd.DataFrame([{"prompt": r["prompt"][1]["content"], "completion": r["completion"][0]["content"]} for r in _judge_rows]
                 ).to_csv(f"{SFT_DIR}/judge_sft_data.csv", index=False)
    print(f"+ {len(_judge_rows)} gold judge-format examples, e.g.:\n{_judge_rows[0]['completion'][0]['content']}")
print(f"{len(sft_dataset)} train ({len(_judge_rows)} of them judge format, {N_SFT_3} 3-number traces) + {len(sft_val_dataset)} val traces; completion tokens median {_ntok.median():.0f}, "
      f"max {_ntok.max()}; dead ends: {pd.Series([t['n_dead_ends'] for t in sft_traces]).value_counts().sort_index().to_dict()}; "
      f"false finishes: {pd.Series([t.get('false_finish') for t in sft_traces]).value_counts().to_dict()}")
print("example:\n" + sft_traces[0]["completion_text"])

+ 3000 gold judge-format examples, e.g.:
46 + 50 = 96
42 * 96 = 4032
4032 / 63 = 64
Answer: (42 * (46 + 50)) / 63
9000 train (3000 of them judge format, 3000 3-number traces) + 200 val traces; completion tokens median 270, max 539; dead ends: {0: 585, 1: 954, 2: 805, 3: 756}; false finishes: {'slip': 452, 'gap': 443}
example:
<reasoning>
Numbers: 46, 42, 63, 50. Target: 64.
Try 46 - 42 = 4. Left: 63, 50, 4.
Then 4 + 50 = 54. Left: 63, 54.
Then 63 + 54 = 64.
Check: 63 + 54 = 117, not 64.
That doesn't work. Back up. Tried: 46 - 42 = 4.
Try 63 * 46 = 2898. Left: 42, 50, 2898.
Then 50 + 42 = 92. Left: 2898, 92.
Closest: 2898 - 92 = 2806, off by 2742.
That doesn't work. Back up. Tried: 46 - 42 = 4; 63 * 46 = 2898.
Try 46 + 50 = 96. Left: 42, 63, 96.
Then 42 * 96 = 4032. Left: 63, 4032.
Then 4032 / 63 = 64. Left: 64.
That reaches 64.
Check: (42 * (46 + 50)) / 63 uses 42, 46, 50, 63 once each and equals 64.
</reasoning>
<solution>
46 + 50 = 96
42 * 96 = 4032
4032 / 63 = 64
</solution>
<answer

In [ ]:
# [SFT train] LoRA SFT of the base model on sft_dataset (prompt-completion: TRL applies chat template with the
# pinned date, one BOS, and puts loss on the assistant turn only, <|eot_id|> included).
# Saves the adapter to SFT_DIR/adapter, validation loss every SFT_EVAL_STEPS, and every SFT_ACC_EVAL_STEPS (and at step
# 0 = the base model) greedy accuracy on the N_SFT_VAL validation puzzles,
# with the kind breakdown. The curve goes into SFT_DIR/acc_curve.csv, completions - into SFT_DIR/acc_eval_stepXXXX.csv.
# Skips when the adapter exists (RETRAIN = True in [config] to train again).
import gc, torch
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, TrainerCallback

SFT_ACC_EVAL_STEPS = 50
_val_puzzles = sft_puzzles[N_SFT:] if not N_SFT_3 else sft_puzzles[N_SFT:N_SFT + 50] + sft3_val[:50]   # 50 of each size

def accuracy_eval(model, step, batch=50, max_new_tokens=768, out_dir=None):
    """Greedy completions for the validation puzzles with the current weights; accuracy, kinds, truncation."""
    was_training = model.training
    model.eval()
    tok.padding_side = "left"
    rows = []
    for b in range(0, len(_val_puzzles), batch):
        ps = _val_puzzles[b:b + batch]
        texts = [tok.apply_chat_template(countdown_prompt(p["numbers"], p["target"]), tokenize=False,
                                         add_generation_prompt=True) for p in ps]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                                 pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for p, o in zip(ps, out[:, enc["input_ids"].shape[1]:]):
            n = int((o != tok.pad_token_id).sum())
            t = tok.decode(o, skip_special_tokens=True)
            truncated = n >= max_new_tokens and tok.eos_token_id not in o.tolist()
            ans = tag(t, "answer")
            rows.append({"step": step, "numbers": json.dumps(p["numbers"]), "target": p["target"], "completion": t,
                         "tokens": n, "truncated": truncated, "answer": ans,
                         "kind": diagnose(ans, p["numbers"], p["target"], truncated)})
    tok.padding_side = "right"
    if was_training:
        model.train()
    df = pd.DataFrame(rows)
    df.to_csv(f"{out_dir or SFT_DIR}/acc_eval_step{step:04d}.csv", index=False)
    return {"step": step, "accuracy": (df.kind == "correct").mean(), "truncated": df.truncated.mean(),
            "tokens": df.tokens.mean(), **{f"kind/{k}": v for k, v in df.kind.value_counts(normalize=True).items()}}

class AccuracyEval(TrainerCallback):
    def __init__(self):
        self.curve = []
    def _run(self, model, step):
        r = accuracy_eval(model, step)
        self.curve.append(r)
        pd.DataFrame(self.curve).to_csv(f"{SFT_DIR}/acc_curve.csv", index=False)
        kinds = ", ".join(f"{k[5:]} {v:.2f}" for k, v in r.items() if k.startswith("kind/") and k != "kind/correct")
        print(f"[accuracy eval] step {step}: accuracy {r['accuracy']:.3f}  truncated {r['truncated']:.2f}  "
              f"tokens {r['tokens']:.0f}  | {kinds}")
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        self._run(model, 0)
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % SFT_ACC_EVAL_STEPS == 0 or state.global_step == state.max_steps:
            self._run(model, state.global_step)

SFT_ADAPTER_DIR = f"{SFT_DIR}/adapter"
if os.path.exists(f"{SFT_ADAPTER_DIR}/adapter_config.json") and not RETRAIN:
    print(f"[SFT train] skipped: {SFT_ADAPTER_DIR} exists (RETRAIN = False)")
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
    model.config.pad_token_id = tok.pad_token_id
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, bias="none", task_type="CAUSAL_LM",
                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    _total_steps = -(-len(sft_dataset) // (SFT_BATCH * SFT_GRAD_ACC)) * SFT_EPOCHS
    sft_args = SFTConfig(
        output_dir=f"{SFT_DIR}/checkpoints",
        num_train_epochs=SFT_EPOCHS, learning_rate=SFT_LR, lr_scheduler_type="cosine",
        warmup_steps=max(1, round(0.05 * _total_steps)),
        per_device_train_batch_size=SFT_BATCH, gradient_accumulation_steps=SFT_GRAD_ACC,
        max_length=1024,                  # prompt (~120 tokens) + trace (a few hundred); nothing is cut
        bf16=True, gradient_checkpointing=True,
        logging_steps=10, save_strategy="no", report_to="none", seed=SEED,
        eval_strategy="steps", eval_steps=SFT_EVAL_STEPS, per_device_eval_batch_size=SFT_BATCH,
    )
    print(f"{len(sft_dataset)} examples, {_total_steps} optimizer steps")
    trainer = SFTTrainer(model=model, args=sft_args, train_dataset=sft_dataset, eval_dataset=sft_val_dataset,
                         processing_class=tok, peft_config=lora_config, callbacks=[AccuracyEval()])

    # Check tokenized examples: one BOS, pinned date, loss only on the completion, ending in <|eot_id|>; a trace starts
    # with <reasoning>, a judge-format example is the neutral system prompt + puzzle -> steps + "Answer: ...".
    _kinds = set()
    for _k in range(min(200, len(trainer.train_dataset))):
        _row = trainer.train_dataset[_k]
        _ids = _row["input_ids"]
        _lab = _row.get("labels") or [i if m else -100 for i, m in zip(_ids, _row["completion_mask"])]
        _comp, _full = tok.decode([i for i, l in zip(_ids, _lab) if l != -100]), tok.decode(_ids)
        assert _ids.count(tok.bos_token_id) == 1 and "26 Jul 2024" in _full
        assert _comp.endswith(tok.eos_token), _comp[-40:]
        if _comp.startswith("<reasoning>"):
            _kinds.add("trace")
        else:
            assert judge_data.NEUTRAL_SYSTEM_PROMPT in _full and "\nAnswer: " in _comp and "<reasoning>" not in _full, _full[:300]
            _kinds.add("judge")
    assert _kinds == ({"trace", "judge"} if N_JUDGE_SFT else {"trace"}), _kinds
    print(f"tokenized ok ({sorted(_kinds)} in the first 200): e.g. {len(_ids)} tokens, {sum(l != -100 for l in _lab)} with loss")
    assert max(len(r["input_ids"]) for r in trainer.train_dataset) <= sft_args.max_length

    trainer.train()
    trainer.model.save_pretrained(SFT_ADAPTER_DIR)   # adapter only: vLLM would load a tokenizer found in the LoRA folder
    _log = pd.DataFrame(trainer.state.log_history)
    _log.to_csv(f"{SFT_DIR}/train_log.csv", index=False)
    print("saved adapter to", SFT_ADAPTER_DIR)
    print(_log.dropna(subset=["eval_loss"])[["step", "eval_loss"]].to_string(index=False))
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

... (9 outputs left out)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


tokenized ok (['judge', 'trace'] in the first 200): e.g. 352 tokens, 207 with loss
[accuracy eval] step 0: accuracy 0.040  truncated 0.50  tokens 457  | no answer (truncated) 0.48, no answer 0.15, wrong numbers (made up or swapped) 0.12, a number reused or added 0.08, right numbers, wrong value 0.07, uses decimals 0.05, a number missing 0.01


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
25,0.653165,0.231126,0.250290,0.926752,134989.000000
50,0.087129,0.075084,0.068594,0.968281,266293.000000
75,0.077679,0.065589,0.066182,0.970300,392369.000000
100,0.071560,0.062599,0.062657,0.971650,519081.000000
125,0.066655,0.060455,0.062576,0.972795,656035.000000
150,0.066863,0.057318,0.058399,0.973173,788667.000000
175,0.063846,0.055894,0.059477,0.973434,926654.000000
200,0.060796,0.055817,0.054297,0.973377,1060244.000000
225,0.060076,0.056173,0.057243,0.973076,1191509.000000
250,0.059762,0.054522,0.054463,0.973950,1324347.000000


[accuracy eval] step 50: accuracy 0.220  truncated 0.03  tokens 223  | right numbers, wrong value 0.42, a number missing 0.14, a number reused or added 0.12, wrong numbers (made up or swapped) 0.07, no answer (truncated) 0.03
[accuracy eval] step 100: accuracy 0.300  truncated 0.22  tokens 457  | a number reused or added 0.29, no answer (truncated) 0.21, right numbers, wrong value 0.15, wrong numbers (made up or swapped) 0.05
[accuracy eval] step 150: accuracy 0.370  truncated 0.19  tokens 382  | right numbers, wrong value 0.38, no answer (truncated) 0.19, a number reused or added 0.05, wrong numbers (made up or swapped) 0.01
[accuracy eval] step 200: accuracy 0.400  truncated 0.23  tokens 475  | no answer (truncated) 0.23, right numbers, wrong value 0.18, a number reused or added 0.15, wrong numbers (made up or swapped) 0.02, a number missing 0.02
[accuracy eval] step 250: accuracy 0.440  truncated 0.35  tokens 520  | no answer (truncated) 0.35, right numbers, wrong value 0.10, a numb

In [ ]:
# [judge DPO training] (JUDGE_DPO only) Continue the SFT adapter with DPO on judge-format pairs (judge_data.py): chosen = the
# gold solution + answer, rejected = a fully procedurally generate wrong solution containing one of the following mistakes:
# swap = one given number replaced by a random unavailable one, every step's arithmetic right, reaches the target;
# slip = one false result at any step, the later steps done right from it, reaches the target;
# miss = every step true, ends off the target (the nearest of 8 random routes). 

# Prompt = the neutral system prompt + puzzle, as the judge sees it. The
# reference is the SFT adapter (TRL copies it into a frozen "ref" adapter). Pairs come from the SFT training puzzles.  Validation pairs from the SFT validation puzzles and 100 more 3-number
# puzzles. Real pairs (N_REAL_4, N_REAL_3; kind "real"): the SFT policy's own right and wrong completions on fresh
# puzzles ([real samples] first).
from peft import PeftModel

LABELS = {"gold_sft_traces": N_SFT + N_SFT_3, "gold_judge_examples": N_JUDGE_SFT,          # shared with every arm (the SFT start)
          "solver_pairs": N_DPO + N_DPO_3,                                       # program-made from the solver
          "checker_real_completions": REAL_LABELS}                               # the policy's samples, graded
if JUDGE_DPO and os.path.isdir(DPO_DIR) and not os.path.exists(f"{DPO_DIR}/labels.json"):
    json.dump(LABELS, open(f"{DPO_DIR}/labels.json", "w"), indent=1)
print("[judge DPO] labels:", LABELS)
if not JUDGE_DPO:
    print("[judge DPO] skipped (JUDGE_DPO = False)")
elif os.path.exists(f"{DPO_ADAPTER_DIR}/adapter_config.json") and not RETRAIN_DPO:
    print(f"[judge DPO] skipped: {DPO_ADAPTER_DIR} exists (RETRAIN_DPO = False)")
else:
    from trl import DPOConfig, DPOTrainer
    assert os.path.exists(f"{SFT_DIR}/adapter/adapter_config.json"), "train the SFT adapter first ([SFT train])"
    _kinds = tuple(DPO_SHARES) if DPO_SHARES else DPO_KINDS
    _syn = lambda pz, seed, n: judge_data.dpo_pairs(pz, random.Random(seed), kinds=_kinds, n=n, shares=DPO_SHARES)
    _pairs = _syn(sft_puzzles[:N_SFT] if DPO_SHARES else sft_puzzles[:min(N_DPO, N_SFT)], SEED + 5, N_DPO)
    _val_pairs = _syn(sft_puzzles[N_SFT:], SEED + 6, N_SFT_VAL)
    if N_DPO_3 and N_SFT_3:   # 3-number puzzles: SFT ones [:N_SFT_3], then the SFT validation ones (DPO validation too)
        _pairs += _syn(dpo3_puzzles[:N_SFT_3] + dpo3_puzzles[N_SFT_3 + N_SFT_VAL:], SEED + 7, N_DPO_3)
        _val_pairs += _syn(sft3_val, SEED + 8, N_SFT_VAL)
    elif N_DPO_3:
        _pairs += _syn(dpo3_puzzles[:N_DPO_3] + (dpo3_puzzles[N_DPO_3 + 100:] if DPO_SHARES else []), SEED + 7, N_DPO_3)
        _val_pairs += _syn(dpo3_puzzles[N_DPO_3:N_DPO_3 + 100], SEED + 8, 100)
    if N_REAL_4 or N_REAL_3:   # chosen = a right, rejected = a wrong completion of the SFT policy, as the reward scores them
        assert os.path.exists(REAL_SAMPLES_PATH), ("no real samples: run [engine] and [real samples], restart the session, "
                                                   "then [config] .. [puzzles] and [judge DPO] (without [engine])")
        _rs = pd.read_csv(REAL_SAMPLES_PATH)
        def _real_pairs(pids, n, rng, samples=SAMPLES):
            """Up to REAL_PER_PUZZLE pairs per puzzle (distinct right and wrong texts at random; only the first `samples`
            completions of each puzzle), until n pairs. Scored
            texts only: a solution of at most 128 tokens and an answer (GRPO: PLAUSIBILITY_MAX_SOLUTION_TOKENS,
            JUDGE_NEEDS_ANSWER); right ones also need no false step (arithmetic_ok not False)."""
            out = []
            for pid in pids:
                if len(out) >= n:
                    break
                right, wrong = set(), set()
                for r in _rs[_rs.puzzle == pid].head(samples).itertuples():
                    if not (isinstance(r.solution, str) and r.solution.strip() and isinstance(r.answer, str) and r.answer.strip()):
                        continue
                    if len(tok(r.solution, add_special_tokens=False)["input_ids"]) > 128:
                        continue
                    text = f"{r.solution.strip()}\nAnswer: {r.answer.strip()}"
                    if not r.correct:
                        wrong.add(text)
                    elif arithmetic_ok(r.solution) is not False:
                        right.add(text)
                k = min(REAL_PER_PUZZLE, len(right), len(wrong))
                p = real_puzzles[pid]
                for c, w in zip(rng.sample(sorted(right), k), rng.sample(sorted(wrong), k)):
                    out.append({"prompt": judge_data.judge_messages(p["numbers"], p["target"]),
                                "chosen": [{"role": "assistant", "content": c}],
                                "rejected": [{"role": "assistant", "content": w}], "kind": "real",
                                "n_numbers": len(p["numbers"])})
            return out[:n]
        for nn, n_real, first, size, (n_use, s_use) in ((4, N_REAL_4, 0, REAL_PUZZLES_4, REAL_USE_4),
                                                         (3, N_REAL_3, REAL_PUZZLES_4, REAL_PUZZLES_3, REAL_USE_3)):
            if n_real:
                _cut = first + size - size // 10                      # the last 10% of the puzzles: validation
                assert n_use <= size - size // 10 and s_use <= SAMPLES, (n_use, s_use)
                _got = _real_pairs(range(first, first + n_use), 10**9 if _real_cut else n_real,
                                   random.Random(SEED + 10 + nn), s_use)
                print(f"real pairs, {nn} numbers: {len(_got)} from {n_use} puzzles x {s_use} completions"
                      + ("" if _real_cut or len(_got) == n_real else
                         f" (of {n_real}: too few puzzles with right and wrong completions: raise REAL_PUZZLES)"))
                _pairs += _got
                _val_pairs += _real_pairs(range(_cut, first + size), 10**9, random.Random(SEED + 20 + nn))
    if N_DPO_3 or N_REAL_4 or N_REAL_3:
        random.Random(SEED + 9).shuffle(_pairs)       # sizes and kinds interleaved in training
    _cols = ("prompt", "chosen", "rejected")
    dpo_train = Dataset.from_list([{k: x[k] for k in _cols} for x in _pairs])
    dpo_val = Dataset.from_list([{k: x[k] for k in _cols} for x in _val_pairs])
    os.makedirs(DPO_DIR, exist_ok=True)
    json.dump(LABELS, open(f"{DPO_DIR}/labels.json", "w"), indent=1)
    pd.DataFrame([{"prompt": x["prompt"][1]["content"], "kind": x["kind"], "n_numbers": x["n_numbers"],
                   "chosen": x["chosen"][0]["content"],
                   "rejected": x["rejected"][0]["content"]} for x in _pairs]).to_csv(f"{DPO_DIR}/dpo_pairs.csv", index=False)
    print(f"{len(_pairs)} pairs ({pd.Series([(x['n_numbers'], x['kind']) for x in _pairs]).value_counts().to_dict()}), "
          f"{len(_val_pairs)} validation pairs; e.g. ({_pairs[0]['kind']})\nCHOSEN\n{_pairs[0]['chosen'][0]['content']}\n"
          f"REJECTED\n{_pairs[0]['rejected'][0]['content']}")

    _base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
    _base.config.pad_token_id = tok.pad_token_id
    dpo_model = PeftModel.from_pretrained(_base, f"{SFT_DIR}/adapter", is_trainable=True)
    _steps = -(-len(dpo_train) // 16) * DPO_EPOCHS
    dpo_args = DPOConfig(
        output_dir=f"{DPO_DIR}/checkpoints", beta=DPO_BETA, learning_rate=DPO_LR, num_train_epochs=DPO_EPOCHS,
        lr_scheduler_type="cosine", warmup_steps=max(1, round(0.05 * _steps)),
        per_device_train_batch_size=4, gradient_accumulation_steps=4, per_device_eval_batch_size=4,
        max_length=512, bf16=True, gradient_checkpointing=True, logging_steps=10, save_strategy="no",
        eval_strategy="steps", eval_steps=50, report_to="none", seed=SEED)
    dpo_trainer = DPOTrainer(model=dpo_model, args=dpo_args, train_dataset=dpo_train, eval_dataset=dpo_val,
                             processing_class=tok)
    assert set(dpo_model.peft_config) == {"default", "ref"}, f"expected a frozen copy of the SFT adapter as reference: {list(dpo_model.peft_config)}"
    print(f"{len(dpo_train)} pairs, {_steps} optimizer steps; reference = the SFT adapter ('ref')")
    dpo_trainer.train()
    dpo_model.save_pretrained(DPO_ADAPTER_DIR, selected_adapters=["default"])   # the trained adapter only (not "ref")
    _log = pd.DataFrame(dpo_trainer.state.log_history)
    _log.to_csv(f"{DPO_DIR}/train_log.csv", index=False)
    print("saved", DPO_ADAPTER_DIR)
    _show = [c for c in ("step", "loss", "eval_loss", "eval_rewards/accuracies", "eval_rewards/margins") if c in _log]
    print(_log[_show].dropna(how="all", subset=_show[1:]).to_string(index=False))

    # check 1: does each judge rank chosen over rejected (mean log-prob of the judged text) on the validation pairs?
    def _mean_lp(model, x, part):
        p_ids = chat_ids(x["prompt"])
        t_ids = tok(x[part][0]["content"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        ids = torch.tensor([p_ids + t_ids], device=model.device)
        with torch.no_grad():
            lp = torch.log_softmax(model(ids).logits[0, :-1].float(), -1).gather(1, ids[0, 1:, None])[:, 0]
        return lp[len(p_ids) - 1:].mean().item()
    dpo_model.eval()
    rows = []
    for name in ("base", "sft", "dpo"):
        for x in _val_pairs:
            if name == "base":
                with dpo_model.disable_adapter():
                    c, r = _mean_lp(dpo_model, x, "chosen"), _mean_lp(dpo_model, x, "rejected")
            else:
                dpo_model.set_adapter("ref" if name == "sft" else "default")
                c, r = _mean_lp(dpo_model, x, "chosen"), _mean_lp(dpo_model, x, "rejected")
            rows.append({"judge": name, "kind": f"{x['n_numbers']}:{x['kind']}", "win": float(c > r) + 0.5 * float(c == r)})
    dpo_model.set_adapter("default")
    _w = pd.DataFrame(rows)
    print("\nvalidation pairs, chosen > rejected by mean log-prob (1.0 = always right; columns n_numbers:kind):")
    print(pd.concat([_w.groupby("judge", sort=False).win.mean().rename("all"),
                     _w.pivot_table(index="judge", columns="kind", values="win", sort=False)], axis=1).round(3))
    # check 2: the policy (same adapter, the policy prompt) after DPO
    _r = accuracy_eval(dpo_model, 0, out_dir=DPO_DIR)
    print(f"policy after DPO, greedy on the {len(_val_puzzles)} validation puzzles: accuracy {_r['accuracy']:.3f}, "
          f"truncated {_r['truncated']:.2f} (SFT adapter at its last [SFT train] check: see SFT_DIR/acc_curve.csv)")
    del dpo_trainer, dpo_model, _base
    gc.collect(); torch.cuda.empty_cache()

[judge DPO] labels: {'gold_sft_traces': 6000, 'gold_judge_examples': 3000, 'solver_pairs': 4500, 'checker_real_completions': 0}
4500 pairs ({(3, 'extra'): 675, (4, 'swap'): 675, (4, 'extra'): 675, (3, 'swap'): 675, (3, 'miss'): 450, (3, 'slip'): 450, (4, 'slip'): 450, (4, 'miss'): 450}), 200 validation pairs; e.g. (extra)
CHOSEN
41 + 71 = 112
112 - 55 = 57
Answer: (41 + 71) - 55
REJECTED
71 + 82 = 153
153 - 41 = 112
112 - 55 = 57
Answer: ((71 + 82) - 41) - 55


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

4500 pairs, 282 optimizer steps; reference = the SFT adapter ('ref')


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.054065,0.044314,0.139270,210219.000000,-0.184088,-0.465484,0.845658,-4.419721,-11.194975,0.985000,6.775254,-48.316165,-135.307954
100,0.008939,0.037496,0.236910,420154.000000,-0.058208,-0.545137,0.809794,-4.753297,-16.618096,0.990000,11.864798,-51.651926,-189.539164
150,0.002058,0.006491,0.302332,630916.000000,0.385557,-0.093087,0.815709,-4.714404,-16.551047,0.995000,11.836643,-51.262991,-188.868671
200,0.001149,0.003482,0.226258,840629.000000,0.450125,0.042009,0.848211,-4.128327,-16.132486,1.000000,12.004159,-45.402224,-184.683064
250,0.001183,0.003802,0.227675,1049703.000000,0.508469,0.140162,0.844170,-4.550742,-16.635145,1.000000,12.084403,-49.626370,-189.709651
282,0.001292,0.003544,0.223976,1181198.000000,0.492799,0.120997,0.844710,-4.494706,-16.612213,1.000000,12.117506,-49.066014,-189.480330


saved /content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000/judge_dpo_swap30extra30slip20miss20_n2250_n3x2250_b0.1_lr2e-05_ep1/adapter
 step     loss  eval_loss  eval_rewards/accuracies  eval_rewards/margins
   10 0.617535        NaN                      NaN                   NaN
   20 0.302553        NaN                      NaN                   NaN
   30 0.073982        NaN                      NaN                   NaN
   40 0.064449        NaN                      NaN                   NaN
   50 0.054065        NaN                      NaN                   NaN
   50      NaN   0.044314                    0.985              6.775254
   60 0.023187        NaN                      NaN                   NaN
   70 0.035656        NaN                      NaN                   NaN
   80 0.063577        NaN                      NaN                   NaN
   90 0.019781        NaN                      NaN           

In [ ]:
# [sample] SAMPLES completions per puzzle (the first EVAL_PUZZLES of each size) to diagnose potential issues (correctness, format, truncation, arithmetic).

os.makedirs(OUT_DIR, exist_ok=True)
SAMPLES_PATH = f"{OUT_DIR}/samples.csv"

def sample_puzzles(pzs):
    _ids = [chat_ids(build_prompt(p["numbers"], p["target"])) for p in pzs]
    _outs = llm.generate([{"prompt_token_ids": x} for x in _ids],
                         SamplingParams(n=SAMPLES, temperature=TEMPERATURE, max_tokens=MAX_TOKENS, seed=SEED),
                         lora_request=POLICY_LORA)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(_outs, _ids)), "vLLM returned outputs for other prompts"
    rows = []
    for pi, (p, o) in enumerate(zip(pzs, _outs)):
        for c in o.outputs:
            t = c.text
            sol, ans = tag(t, "solution"), tag(t, "answer")
            rows.append({"puzzle": pi, "numbers": json.dumps(p["numbers"]), "target": p["target"],
                         "n_solutions": p["n_solutions"], "completion": t, "tokens": len(c.token_ids),
                         "truncated": c.finish_reason == "length",
                         "format_ok": bool(re.search(r"<reasoning>.*</reasoning>\s*<solution>.*</solution>\s*"
                                                     r"<answer>.*</answer>", t, re.DOTALL)),
                         "solution": sol, "answer": ans,
                         "correct": check_answer(ans, p["numbers"], p["target"]) if ans else False,
                         "arith_ok": arithmetic_ok(sol)})
    return pd.DataFrame(rows)

samples_by_size = {}
for _n, _pz in puzzles_by_size.items():   # 4 = the test puzzles (samples.csv); 3 and 5 = the other sizes
    _path = SAMPLES_PATH if _n == 4 else f"{OUT_DIR}/samples_n{_n}.csv"
    if os.path.exists(_path):
        samples_by_size[_n] = pd.read_csv(_path)
        print(f"loaded {_path}")
    else:
        samples_by_size[_n] = sample_puzzles(_pz[:EVAL_PUZZLES])
        samples_by_size[_n].to_csv(_path, index=False)
        print(f"saved {_path}")
for _n, _df in samples_by_size.items():
    _df["kind"] = [diagnose(a, json.loads(n), t, tr) for a, n, t, tr in
                   zip(_df.answer, _df.numbers, _df.target, _df.truncated)]
    assert (_df.kind == "correct").equals(_df.correct.astype(bool)), "diagnose and the checker disagree"
    if _n != 4:
        print(f"{_n}-number: accuracy {_df.correct.mean():.3f}  truncated {_df.truncated.mean():.3f}  tokens {_df.tokens.mean():.0f}")
samples = samples_by_size[4]
_pass = samples.groupby("puzzle").correct.mean()
print(f"accuracy {samples.correct.mean():.3f}  format {samples.format_ok.mean():.3f}  truncated {samples.truncated.mean():.3f}  "
      f"tokens {samples.tokens.mean():.0f}")
print(f"puzzles with correct and wrong completions: {((_pass > 0) & (_pass < 1)).sum()} of {len(_pass)}  "
      f"(all wrong {(_pass == 0).sum()}, all correct {(_pass == 1).sum()})")
print("what the answers are:", samples.kind.value_counts().to_dict())
print("arithmetic of solution lines:", samples.arith_ok.map({True: "ok", False: "error", None: "unknown"})
      .fillna("unknown").value_counts().to_dict())

Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1600/1600 [02:53<00:00,  9.25it/s, est. speed input: 1368.55 toks/s, output: 6095.00 toks/s]


saved /content/drive/MyDrive/Consistency/countdown/disc/Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000-user-n100x16-t0.9-seed0/samples.csv


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1600/1600 [01:25<00:00, 18.70it/s, est. speed input: 2711.04 toks/s, output: 6409.44 toks/s]


saved /content/drive/MyDrive/Consistency/countdown/disc/Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000-user-n100x16-t0.9-seed0/samples_n3.csv


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1600/1600 [03:14<00:00,  8.21it/s, est. speed input: 1240.03 toks/s, output: 5979.22 toks/s]


saved /content/drive/MyDrive/Consistency/countdown/disc/Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000-user-n100x16-t0.9-seed0/samples_n5.csv
3-number: accuracy 0.734  truncated 0.033  tokens 343
5-number: accuracy 0.126  truncated 0.267  tokens 728
accuracy 0.194  format 0.780  truncated 0.215  tokens 659
puzzles with correct and wrong completions: 59 of 100  (all wrong 41, all correct 0)
what the answers are: {'right numbers, wrong value': 703, 'no answer (truncated)': 343, 'correct': 310, 'a number reused or added': 120, 'a number missing': 64, 'wrong numbers (made up or swapped)': 52, 'no answer': 6, 'uses decimals': 2}
arithmetic of solution lines: {'error': 649, 'unknown': 476, 'ok': 475}
